In [ ]:
# ==============================================================================
# Cell 1: Load and Split the Dataset
# ==============================================================================
import json
import os

# --- Configuration ---
# 1. Make sure to upload your dataset to Colab's file system first.
# 2. Change this filename to match your uploaded file.
original_dataset_filename = '/kaggle/input/prepare-dataset/prepare_Dataset.json' # <-- IMPORTANT: Change if your filename is different

# Define the output filenames
knowledge_base_filename = 'knowledge_base.json'
validation_set_filename = 'validation_set.json'

# --- Main Logic ---
print("--- Step 1: Data Splitting ---")

# Check if the original dataset file exists
if not os.path.exists(original_dataset_filename):
    print(f"❌ ERROR: The file '{original_dataset_filename}' was not found.")
    print("Please make sure you have uploaded the dataset and the filename is correct.")
else:
    # Load the entire dataset from the JSON file
    with open(original_dataset_filename, 'r') as f:
        all_data = json.load(f)
    print(f"✅ Successfully loaded {len(all_data)} total records.")

    # Calculate the split point (80% of the data)
    total_size = len(all_data)
    split_point = int(total_size * 0.8)

    # Split the data sequentially (no shuffling, as requested)
    knowledge_base_data = all_data[:split_point]
    validation_set_data = all_data[split_point:]

    # Verify the sizes
    print(f"Splitting data...")
    print(f"  - Knowledge Base size (80%): {len(knowledge_base_data)} records")
    print(f"  - Validation Set size (20%): {len(validation_set_data)} records")

    # Save the knowledge base to a new JSON file
    with open(knowledge_base_filename, 'w') as f:
        json.dump(knowledge_base_data, f, indent=4)
    print(f"✅ Knowledge Base saved to '{knowledge_base_filename}'")

    # Save the validation set to a new JSON file
    with open(validation_set_filename, 'w') as f:
        json.dump(validation_set_data, f, indent=4)
    print(f"✅ Validation Set saved to '{validation_set_filename}'")

    print("\n--- Data split complete. ---")

In [ ]:
!pip3 install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
!pip install numpy==1.26.4 sentence-transformers==2.7.0 chromadb==0.5.0 tqdm --upgrade transformers accelerate bitsandbytes -q

In [3]:
import json
import torch
from sentence_transformers import SentenceTransformer
import chromadb

# --- 2. Configuration ---
embedding_model_name = 'BAAI/bge-large-en-v1.5'
knowledge_base_filename = '/kaggle/working/knowledge_base.json'
validation_set_filename = '/kaggle/working/validation_set.json'
validation_with_embeddings_filename = 'validation_set_with_embeddings.json'
collection_name = "bdd_scenarios_kb"

# --- 3. Load Embedding Model ---
print("--- Initializing Models and Database ---")

# Check for GPU availability and set the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Using device: {device}")

# Load the sentence-transformer model. This may take a minute to download.
print(f"⏳ Loading embedding model '{embedding_model_name}'...")
embedding_model = SentenceTransformer(embedding_model_name, device=device)
print("✅ Embedding model loaded successfully.")

# --- 4. Setup Chroma DB ---
# Initialize the Chroma DB client. We'll use an in-memory/on-disk version.
client = chromadb.Client()

# Create a new collection. If it already exists, this will load the existing one.
collection = client.get_or_create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"} # Use cosine similarity
)
print(f"✅ Chroma DB collection '{collection_name}' is ready.")
print("\n--- Setup complete. Proceed to the next cell to index the knowledge base. ---")

--- Initializing Models and Database ---
✅ Using device: cuda
⏳ Loading embedding model 'BAAI/bge-large-en-v1.5'...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


✅ Embedding model loaded successfully.


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Chroma DB collection 'bdd_scenarios_kb' is ready.

--- Setup complete. Proceed to the next cell to index the knowledge base. ---


In [ ]:
# ==============================================================================
# Cell 3: Index the Knowledge Base (Part 2a)
# ==============================================================================
from tqdm.auto import tqdm # For nice progress bars

print("--- Part 2a: Processing and Indexing the Knowledge Base ---")

# Load the knowledge base data
with open(knowledge_base_filename, 'r') as f:
    knowledge_base_data = json.load(f)

# Prepare lists for batch processing
ids_to_add = []
records_to_embed = []
metadata_to_add = []

print("⏳ Preparing knowledge base records...")
for i, item in enumerate(knowledge_base_data):
    # Dynamically find the keys for the record and feature
    record_key = next((key for key in item if key.startswith('Record')), None)
    feature_key = next((key for key in item if key.startswith('Feature')), None)

    if record_key and feature_key:
        record_text = item[record_key]
        bdd_scenario_text = item[feature_key]

        # Add data to our lists for batch processing
        records_to_embed.append(record_text)
        ids_to_add.append(f"kb_record_{i}")
        metadata_to_add.append({
            "original_record": record_text,
            "bdd_scenario": bdd_scenario_text
        })

# Create embeddings in a single batch (much faster)
print(f"⏳ Creating embeddings for {len(records_to_embed)} knowledge base records... (This may take a moment)")
embeddings_to_add = embedding_model.encode(
    records_to_embed,
    batch_size=32, # Adjust batch size based on your GPU memory
    show_progress_bar=True,
    convert_to_tensor=False
)

# Add the data to Chroma DB in one go
print("⏳ Adding all records to Chroma DB collection...")
collection.add(
    ids=ids_to_add,
    embeddings=embeddings_to_add.tolist(), # Convert numpy array to list
    metadatas=metadata_to_add
)

print(f"✅ Successfully indexed {collection.count()} records into the knowledge base.")
print("\n--- Knowledge base indexing complete. ---")


In [ ]:
# ==============================================================================
# Cell 4: Prepare the Validation Set (Part 2b)
# ==============================================================================
from tqdm.auto import tqdm

print("--- Part 2b: Processing the Validation Set ---")

# Load the validation set data
with open(validation_set_filename, 'r') as f:
    validation_set_data = json.load(f)

validation_records_to_embed = []
enhanced_validation_data = []

print("⏳ Preparing validation set records...")
for item in validation_set_data:
    record_key = next((key for key in item if key.startswith('Record')), None)
    feature_key = next((key for key in item if key.startswith('Feature')), None)

    if record_key and feature_key:
        record_text = item[record_key]
        bdd_scenario_text = item[feature_key]
        validation_records_to_embed.append(record_text)
        # Store the plain text data in our new structure
        enhanced_validation_data.append({
            "record_text": record_text,
            "golden_scenario": bdd_scenario_text
        })

# Create embeddings for the validation set in a single batch
print(f"⏳ Creating embeddings for {len(validation_records_to_embed)} validation records...")
validation_embeddings = embedding_model.encode(
    validation_records_to_embed,
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=False
)

# Add the newly created embeddings to our enhanced data structure
for i, item in enumerate(enhanced_validation_data):
    item['embedding'] = validation_embeddings[i].tolist()

# Save the processed validation set to a new file
with open(validation_with_embeddings_filename, 'w') as f:
    json.dump(enhanced_validation_data, f, indent=4)

print(f"✅ Validation set processed and saved with embeddings to '{validation_with_embeddings_filename}'")
print("\n--- Validation set preparation complete. ---")


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
hf_token= user_secrets.get_secret("HF_demo")


# Log in to Hugging Face
login(token=hf_token)

In [ ]:
import torch
import transformers

model_id = "meta-llama/Llama-3.1-8B"

print(f"--- Loading Llama 3.1 model into a pipeline: {model_id} ---")
print("This will take several minutes and use a significant amount of RAM...")

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    # CHANGE THIS LINE: Use float16 for T4/P100 compatibility
    model_kwargs={"torch_dtype": torch.float16}, 
    device_map="auto"
)
print("✅ Llama 3.1 pipeline ready.")

In [ ]:
# ==============================================================================
# Cell 1: Master Installation (Force Upgrade for Security Fix)
# ==============================================================================
# This upgrades PyTorch to fix the 'CVE-2025-32434' ValueError.

!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install numpy==1.26.4 sentence-transformers==2.7.0 chromadb==0.5.0 transformers==4.42.3 accelerate bitsandbytes tqdm evaluate -q

print("✅ Libraries updated to PyTorch 2.6+!")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_id = "Salesforce/codet5p-770m"

print(f"--- Loading model: {model_id} ---")

# 1. Load Tokenizer explicitly
tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    trust_remote_code=True
)

# 2. Load Model explicitly
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 3. Create Pipeline using the loaded model/tokenizer
pipeline = pipeline(
    "text2text-generation", 
    model=model, 
    tokenizer=tokenizer
)

print("✅ CodeT5+ pipeline ready.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_id = "Salesforce/codet5p-220m"

print(f"--- Loading model: {model_id} ---")

# 1. Load Tokenizer explicitly
tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    trust_remote_code=True
)

# 2. Load Model explicitly
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 3. Create Pipeline using the loaded model/tokenizer
pipeline = pipeline(
    "text2text-generation", 
    model=model, 
    tokenizer=tokenizer
)

print("✅ CodeT5+ pipeline ready.")

In [ ]:
#Ensure u restart
!pip install -U bitsandbytes

In [ ]:
#Not sure about this one
# 1. Uninstall existing conflicting versions
!pip uninstall -y torch torchvision torchaudio torch_xla

# 2. Install matching stable versions from PyPI
!pip install torch torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html

In [ ]:
# ==============================================================================
# TPU Installation Script
# ==============================================================================
# This script installs the necessary libraries to run PyTorch on a TPU.

!pip install torch~=2.3.0 torch_xla~=2.3.0 torchvision~=0.18.0 -f https://storage.googleapis.com/libtpu-releases/index.html

# Now install the other libraries
!pip install "transformers>=4.42.0" datasets evaluate sentence-transformers chromadb tqdm -q

print("\n✅ PyTorch/XLA installation complete.")

In [1]:
# ==============================================================================
# Optimized Model Loading Script (for DeepSeek Coder 33B with Quantization)
# ==============================================================================
# This script uses 4-bit quantization with bitsandbytes to load the large
# 33B model efficiently on the available Kaggle GPUs.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

model_id = "deepseek-ai/deepseek-coder-33b-base"

print(f"--- Loading DeepSeek Coder 33B Instruct model with 4-bit quantization: {model_id} ---")
print("This will still take several minutes, but will be much more memory-efficient.")

# --- THE OPTIMIZATION: Configuration for 4-bit quantization ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16 # Use float16 for T4 compatibility
)
# -------------------------------------------------------------

# --- V1 models REQUIRE trust_remote_code=True ---
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

# --- Load the model with the quantization config ---
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bnb_config, # <-- APPLIED THE QUANTIZATION
    device_map="auto" # This will now place the smaller model entirely on your GPUs
)
# --------------------------------------------------

pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)
tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

print("✅ DeepSeek Coder 33B Instruct pipeline ready (loaded in 4-bit).")

2026-01-07 16:22:57.373826: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767802977.398075     205 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767802977.405520     205 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

model-00007-of-00007.safetensors:   0%|          | 0.00/7.38G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Device set to use cuda:0


✅ DeepSeek Coder 33B Instruct pipeline ready (loaded in 4-bit).


In [ ]:
!pip install openai -q

In [ ]:
pip install evaluate scikit-learn

In [4]:
# ==============================================================================
# Cell 10: Final Full Evaluation (Crash-Proof Version)
# ==============================================================================
# This version includes a safety valve for Unicode errors so the script
# will simply skip weird characters instead of crashing.

import json
from tqdm.auto import tqdm
import evaluate
import numpy as np
import gc
import torch
import os

# --- 1. Required Helper Functions ---

def build_prompt(task_record, retrieved_examples):
    prompt = """1.Your role:
You are an expert Quality Assurance engineer specializing in writing Behavior-driven Development (BDD) scenarios in Gherkin syntax.
2.Your task:
Your task is to translate the given "Source Record" into a compliant and accurate “BDD Scenario”. Use the provided examples to understand the correct format and style.
After you have completed the BDD Scenario for the Task, you must stop. Do not generate any further examples or text.
"""
    for i, example in enumerate(retrieved_examples):
        record, scenario = example['original_record'], example['bdd_scenario']
        example_dict = {"Source Record": record, "BDD Scenario": scenario}
        prompt += f"\nExample{i+1}:\n{json.dumps(example_dict, indent=4)}\n"

    task_dict = {"Source Record": task_record}
    task_json_str = json.dumps(task_dict, indent=4).strip()[:-1].strip()
    prompt += f"""
Task:
{task_json_str},
    "BDD Scenario": """
    return prompt

def calculate_metrics(predictions, references, tokenizer):
    print("\n--- Calculating Performance Metrics ---")
    em_score = evaluate.load("exact_match").compute(references=references, predictions=predictions)['exact_match']
    bleu_score = evaluate.load("bleu").compute(predictions=[p if p.strip() else " " for p in predictions], references=[[r] for r in references])['bleu']
    f1_scores = []
    for pred, ref in zip(predictions, references):
        pred_tokens, ref_tokens = set(tokenizer.encode(pred)), set(tokenizer.encode(ref))
        common_tokens = pred_tokens.intersection(ref_tokens)
        if not common_tokens: f1_scores.append(0.0); continue
        precision = len(common_tokens) / len(pred_tokens) if pred_tokens else 0
        recall = len(common_tokens) / len(ref_tokens) if ref_tokens else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_scores.append(f1)
    f1_score = np.mean(f1_scores)
    return {"f1_score": f1_score, "bleu_score": bleu_score, "exact_match_score": em_score}

# --- 2. Load Full Validation Data ---
with open('validation_set_with_embeddings.json', 'r') as f:
    full_validation_data = json.load(f) #<-adjust this

# --- 3. Checkpointing Setup ---
checkpoint_filename = 'evaluation_results_checkpoint.json'
report_data = []

# Resume from crash if possible
if os.path.exists(checkpoint_filename):
    print(f"🔄 Found existing checkpoint file. Resuming progress.")
    with open(checkpoint_filename, 'r') as f:
        report_data = json.load(f)
    start_index = len(report_data)
else:
    print(f"🏁 Starting a new evaluation run.")
    start_index = 0

records_to_process = full_validation_data[start_index:]
print(f"--- Starting FULL EVALUATION for {len(records_to_process)} remaining records ---")

# --- 4. The Main Evaluation Loop ---
if records_to_process:
    for i, item in enumerate(tqdm(records_to_process, desc="Evaluating Full Set")):
        
        # RAG Retrieval
        results = collection.query(query_embeddings=[item['embedding']], n_results=3) #To adjust number of shots
        final_prompt = build_prompt(item['record_text'], results['metadatas'][0])
        
        outputs = pipeline(final_prompt, max_new_tokens=512, eos_token_id=pipeline.tokenizer.eos_token_id, do_sample=False, pad_token_id=pipeline.tokenizer.eos_token_id)

        response_text = outputs[0]['generated_text']
        #generated_scenario_raw = response_text[len(final_prompt):].strip() -<for llama 3.1-8b
        generated_scenario_raw= response_text.strip()
        # Basic cleanup
        if generated_scenario_raw.startswith('"'): generated_scenario_raw = generated_scenario_raw[1:]
        if generated_scenario_raw.endswith('"}'): generated_scenario_raw = generated_scenario_raw[:-2]
        if generated_scenario_raw.endswith('"'): generated_scenario_raw = generated_scenario_raw[:-1]
        
        # --- THE FIX: SAFETY VALVE FOR UNICODE ERRORS ---
        try:
            generated_scenario = generated_scenario_raw.encode().decode('unicode_escape')
        except Exception:
            # If decoding fails (e.g. trailing backslash), use the raw text
            generated_scenario = generated_scenario_raw
        # ------------------------------------------------

        report_data.append({
            "input_record": item['record_text'],
            "expected_scenario": item['golden_scenario'],
            "generated_scenario": generated_scenario
        })

        del results, final_prompt, outputs, response_text
        gc.collect()
        torch.cuda.empty_cache()

        # Save progress every 25 records
        if (i + 1) % 25 == 0 or (i + 1) == len(records_to_process):
            with open(checkpoint_filename, 'w') as f:
                json.dump(report_data, f, indent=4)
            # tqdm.write(f"💾 Checkpoint saved.")

print("✅ Full evaluation run complete.")

# --- 5. Calculate Metrics ---
predictions = [d['generated_scenario'] for d in report_data]
references = [d['expected_scenario'] for d in report_data]
metrics = calculate_metrics(predictions, references, pipeline.tokenizer)

# --- 6. Generate Final Report ---
report_filename = "final_evaluation_report.txt"
report_lines = []

for data in report_data:
    report_lines.append("Input Record:\n")
    report_lines.append(data['input_record'])
    report_lines.append("\n\nExpected BDD scenario:\n")
    report_lines.append(data['expected_scenario'])
    report_lines.append("\n\nGenerated BDD scenario:\n")
    report_lines.append(data['generated_scenario'])
    report_lines.append("\n" + "="*50 + "\n")

report_lines.append("\n. . .\n\n")
report_lines.append(f"F1 Score (Token-based): {metrics['f1_score']:.4f}")
report_lines.append(f"BLEU Score:               {metrics['bleu_score']:.4f}")
report_lines.append(f"Exact Match Score:        {metrics['exact_match_score']:.4f}")

final_report = "\n".join(report_lines)
with open(report_filename, 'w') as f:
    f.write(final_report)

print("\n--- FINAL REPORT SUMMARY ---")
print(f"F1 Score: {metrics['f1_score']:.4f}")
print(f"BLEU Score: {metrics['bleu_score']:.4f}")
print(f"Exact Match: {metrics['exact_match_score']:.4f}")
print(f"\n✅ Detailed report saved to '{report_filename}'")

🔄 Found existing checkpoint file. Resuming progress.
--- Starting FULL EVALUATION for 95 remaining records ---


Evaluating Full Set:   0%|          | 0/95 [00:00<?, ?it/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✅ Full evaluation run complete.

--- Calculating Performance Metrics ---



--- FINAL REPORT SUMMARY ---
F1 Score: 0.2723
BLEU Score: 0.0492
Exact Match: 0.0000

✅ Detailed report saved to 'final_evaluation_report.txt'


In [5]:
rm /kaggle/working/final_evaluation_report.txt

In [6]:
rm /kaggle/working/evaluation_results_checkpoint.json

In [ ]:
rm -rf /kaggle/working/*